In [1]:
import pandas as pd
import polars as pl
from pathlib import Path
import util

pd.set_option('display.float_format', '{:,.0f}'.format)

In [2]:
def mic_equity_geog(df_in, equity_group):

    df_inc = df.pivot_table(
        index='person_work_mic',
        columns=equity_group,
        values='psexpfac',
        aggfunc='sum'
    ).reset_index()

    df_inc = df_inc.rename_axis(None, axis=1)
    pd.set_option('display.float_format', '{:,.0f}'.format)
    df_inc['Percent in Equity Geog'] = df_inc[1] / (df_inc[0] + df_inc[1])
    df_inc['Percent in Equity Geog'] = df_inc['Percent in Equity Geog'].fillna(0)
    df_inc['Percent in Equity Geog'] = (df_inc['Percent in Equity Geog'] * 100).round(1).astype(str) + '%'
    df_inc.sort_values(by=0, ascending=False)

    df_inc.rename(columns={'person_work_mic': 'MIC Work Location',
                        0: 'Below Regional Average', 
                                1: 'Above Regional Average', 
                                2: 'Higher Share of Equity Population'}, inplace=True)
    df_inc.index = df_inc['MIC Work Location']
    df_inc.drop('MIC Work Location', axis=1, inplace=True)
    # Percent of people of color
    
    return df_inc

## Workers in Manufacturing-Industrial Centers (MICs)

In [3]:
df = pd.read_csv(util.output_path / 'agg/dash/mic_workers.csv')

In [4]:
df = pd.read_csv(util.output_path / 'agg/dash/mic_workers.csv')
df = df[df['pwtyp'].isin(['Paid Full-Time Worker', 'Paid Part-Time Worker'])]
df.loc[df['person_work_mic'].isnull(), 'person_work_mic'] = 'Outside MIC'
_df = df[['psexpfac','person_work_mic']].groupby('person_work_mic').sum()[['psexpfac']]
_df.index
_df.rename(columns={'psexpfac': 'Total Workers'}, inplace=True)
_df


,Total Workers
person_work_mic,
Ballard-Interbay,"17,207"
Cascade,"9,048"
Duwamish,"61,706"
Frederickson,"3,536"
Kent MIC,"43,987"
North Tukwila,"6,702"
Outside MIC,"2,077,333"
Paine Field / Boeing Everett,"36,184"
Port of Tacoma,"8,797"


In [5]:
mic_equity_geog(df, "hh_efa_pov200").loc[_df.index]

,Below Regional Average,Above Regional Average,Higher Share of Equity Population,Percent in Equity Geog
person_work_mic,,,,
Ballard-Interbay,"12,376","3,210","1,621",20.6%
Cascade,"5,127","2,765","1,156",35.0%
Duwamish,"36,623","15,954","9,129",30.3%
Frederickson,"1,815","1,229",492,40.4%
Kent MIC,"21,958","13,109","8,920",37.4%
North Tukwila,"3,296","2,047","1,359",38.3%
Outside MIC,"1,295,560","519,552","262,221",28.6%
Paine Field / Boeing Everett,"20,963","9,857","5,364",32.0%
Port of Tacoma,"3,899","3,012","1,886",43.6%


In [6]:
mic_equity_geog(df, "hh_efa_poc").loc[_df.index]

,Below Regional Average,Above Regional Average,Higher Share of Equity Population,Percent in Equity Geog
person_work_mic,,,,
Ballard-Interbay,"10,985","4,053","2,169",27.0%
Cascade,"7,951",943,154,10.6%
Duwamish,"28,057","17,444","16,205",38.3%
Frederickson,"1,914","1,094",528,36.4%
Kent MIC,"15,588","13,191","15,208",45.8%
North Tukwila,"2,360","1,915","2,427",44.8%
Outside MIC,"1,120,718","608,877","347,738",35.2%
Paine Field / Boeing Everett,"20,445","12,640","3,099",38.2%
Port of Tacoma,"4,145","2,926","1,726",41.4%


In [7]:
mic_equity_geog(df, "hh_efa_lep").loc[_df.index]

,Below Regional Average,Above Regional Average,Higher Share of Equity Population,Percent in Equity Geog
person_work_mic,,,,
Ballard-Interbay,"11,888","2,957","2,362",19.9%
Cascade,"7,722",910,416,10.5%
Duwamish,"33,078","13,528","15,100",29.0%
Frederickson,"2,753",495,288,15.2%
Kent MIC,"18,207","11,158","14,622",38.0%
North Tukwila,"2,879","1,583","2,240",35.5%
Outside MIC,"1,294,616","438,989","343,728",25.3%
Paine Field / Boeing Everett,"19,461","8,629","8,094",30.7%
Port of Tacoma,"5,857","1,719","1,221",22.7%


## Commute Characteristics

In [8]:
df = pd.read_csv(util.output_path / 'agg/dash/tour_mic_dest.csv')
# Commute tours to MICs
df = df[df['pdpurp']=="Work"]
df.loc[df['tour_d_mic'].isnull(), 'tour_d_mic'] = 'Outside MIC'

# tmodetp by tour_d_mic
df_mode = pd.pivot_table(
    df,
    index='tour_d_mic',
    columns='tmodetp',
    values='toexpfac',
    aggfunc='sum'
)

# mode share by tour_d_mic
df_mode = df_mode.div(df_mode.sum(axis=1), axis=0)
df_mode = df_mode.fillna(0)
df_mode = df_mode.reset_index()
df_mode.rename(columns={'tour_d_mic': 'MIC Work Location'}, inplace=True)
df_mode.index = df_mode['MIC Work Location']
df_mode.drop('MIC Work Location', axis=1, inplace=True)

df_mode['Transit'] = df_mode['Transit']+df_mode['Park']
df_mode['HOV'] = df_mode['HOV2'] + df_mode['HOV3+']
if util.input_config["include_tnc"] == True:
    df_mode['Walk/Bike/Other'] =  df_mode['Walk']+df_mode['Bike']+df_mode['TNC']
else:
    df_mode['Walk/Bike/Other'] =  df_mode['Walk'] + df_mode['Bike']
# df_mode = df_mode.sort_values(by='Drive Alone', ascending=False)
# Show results as percentages
df_mode = df_mode.applymap(lambda x: f"{x:.1%}")
# df_mode.sort_values(by='Walk', ascending=False, inplace=True)


df_mode[['SOV', 'HOV', 'Transit', 'Walk/Bike/Other']]

tmodetp,SOV,HOV,Transit,Walk/Bike/Other
MIC Work Location,,,,
Ballard-Interbay,65.6%,25.7%,1.9%,6.8%
Cascade,67.6%,30.4%,0.1%,1.9%
Duwamish,66.2%,29.0%,3.0%,1.8%
Frederickson,66.9%,32.0%,0.0%,1.2%
Kent MIC,66.6%,31.1%,1.4%,0.9%
North Tukwila,67.4%,29.8%,1.6%,1.1%
Outside MIC,62.3%,27.6%,3.8%,6.4%
Paine Field / Boeing Everett,68.9%,29.4%,0.3%,1.3%
Port of Tacoma,67.4%,31.2%,0.4%,1.0%


In [9]:
# Commute distance
pd.set_option('display.float_format', '{:,.1f}'.format)

df = pd.read_csv(util.output_path / 'agg/dash/tour_distance_mic.csv')
df = df[df['pdpurp'] == 'Work']
df['wt_dist'] = df['tautodist_bin'] * df['toexpfac']
df = df.groupby("person_work_mic").agg(
    {'wt_dist': 'sum', 'toexpfac': 'sum'}
)
df['average_distance'] = df['wt_dist']/df['toexpfac']
df[['average_distance']]

,average_distance
person_work_mic,
Ballard-Interbay,10.3
Cascade,10.6
Duwamish,12.9
Frederickson,8.4
Kent MIC,12.6
North Tukwila,12.8
Paine Field / Boeing Everett,11.0
Port of Tacoma,11.4
Puget Sound Industrial Center- Bremerton,13.2


## Demographics

In [10]:
# Demographics 
df = pd.read_csv(util.output_path / 'agg/dash/mic_workers.csv')
df['income_wt'] = df['hhincome_thousands'] * df['psexpfac']
df['income_wt'] = df['income_wt'].fillna(0)
df = df.groupby('person_work_mic').agg(
    {'psexpfac': 'sum', 'income_wt': 'sum'}
).reset_index()
df["avg_weighted_hh_income"] = df['income_wt']/df['psexpfac']
df['avg_weighted_hh_income'] = df['avg_weighted_hh_income'].apply(lambda x: f"${x:,.0f}")
df[['person_work_mic', 'avg_weighted_hh_income']]

,person_work_mic,avg_weighted_hh_income
0,Ballard-Interbay,"$189,847"
1,Cascade,"$150,758"
2,Duwamish,"$182,620"
3,Frederickson,"$139,941"
4,Kent MIC,"$160,805"
5,North Tukwila,"$163,396"
6,Paine Field / Boeing Everett,"$157,623"
7,Port of Tacoma,"$141,303"
8,Puget Sound Industrial Center- Bremerton,"$131,919"
9,Sumner Pacific,"$149,470"


In [11]:
df = pd.read_csv(util.output_path / 'agg/dash/mic_workers.csv')
df['hhsize_wt'] = df['hhsize'] * df['psexpfac']
df['hhsize_wt'] = df['hhsize_wt'].fillna(0)
df = df.groupby('person_work_mic').agg(
    {'psexpfac': 'sum', 'hhsize_wt': 'sum'}
)
df['avg_wt_hhsize'] = df['hhsize_wt']/df['psexpfac']
df = df.reset_index()
df[['person_work_mic', 'avg_wt_hhsize']]

,person_work_mic,avg_wt_hhsize
0,Ballard-Interbay,2.9
1,Cascade,3.5
2,Duwamish,3.1
3,Frederickson,3.6
4,Kent MIC,3.4
5,North Tukwila,3.3
6,Paine Field / Boeing Everett,3.2
7,Port of Tacoma,3.4
8,Puget Sound Industrial Center- Bremerton,3.0
9,Sumner Pacific,3.4


In [12]:
df = pd.read_csv(util.output_path / 'agg/dash/mic_workers.csv')
# Pivot table of worker type by person_work_mic
df_worker_type = df.pivot_table(
    index='person_work_mic',
    columns='pwtyp',
    values='psexpfac',
    aggfunc='sum'
)

# Share by worker type

df_worker_type = df_worker_type.div(df_worker_type.sum(axis=1), axis=0)
df_worker_type.rename(columns={'Paid Full-Time Worker': 'Full Time', 'Paid Part-Time Worker': 'Part Time'})
df_worker_type.applymap(lambda x: f"{x:.1%}")

pwtyp,Paid Full-Time Worker,Paid Part-Time Worker
person_work_mic,,
Ballard-Interbay,83.0%,17.0%
Cascade,73.7%,26.3%
Duwamish,87.0%,13.0%
Frederickson,67.1%,32.9%
Kent MIC,85.1%,14.9%
North Tukwila,80.3%,19.7%
Paine Field / Boeing Everett,83.5%,16.5%
Port of Tacoma,82.2%,17.8%
Puget Sound Industrial Center- Bremerton,67.4%,32.6%
